# ChargeCast — Data quality and exploratory analysis

Review notebook for Clean v1. One row represents a charging device (`com_id`) on a date recorded in the source. The objective is to forecast the next day's recorded session count.

Keep missing labels separate from inferred zeros. The cohort was selected retrospectively, timestamps have an unverified timezone and the available labels depend on coverage assumptions. This notebook checks internal consistency and profiles the input to the forecasting experiment.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Resolve paths when Jupyter starts at the repo root or in notebooks/.
working_dir = Path.cwd().resolve()
relative_panel = Path("data/processed/daily_panel.csv.gz")
repo = next(
    (folder for folder in (working_dir, *working_dir.parents)
     if (folder / relative_panel).is_file()),
    None,
)
if repo is None:
    raise FileNotFoundError("Open this notebook from the ChargeCast repository.")

panel = pd.read_csv(
    repo / relative_panel,
    dtype={"com_id": "string", "sessions_target": "Int64"},
    parse_dates=["reported_date"],
).sort_values(["com_id", "reported_date"]).reset_index(drop=True)

audit = json.loads((repo / "reports/data_quality/audit_summary.json").read_text())
print(f"{len(panel):,} device-days | {panel.com_id.nunique():,} devices")
print(f"{panel.reported_date.min():%Y-%m-%d} to {panel.reported_date.max():%Y-%m-%d}")

## 1. Check the daily panel

Check unique device-date keys, a continuous calendar, label validity and agreement with the Clean v1 audit. Dates with masked labels stay in the calendar so that a seven-day lag always points to seven calendar days earlier.

In [ ]:
key = ["com_id", "reported_date"]
assert panel[key].notna().all().all(), "Missing device ID or date"
assert not panel.duplicated(key).any(), "Duplicate device-date key"
date_steps = panel.groupby("com_id", sort=False).reported_date.diff().dropna()
assert date_steps.eq(pd.Timedelta(days=1)).all(), "Calendar discontinuity"
dates_per_device = panel.groupby("com_id", sort=False).reported_date.size()
assert dates_per_device.eq(panel.reported_date.nunique()).all(), "Incomplete device calendar"

valid = panel["target_valid"]
target = panel["sessions_target"]
assert target.notna().equals(valid), "Label and validity flag disagree"
assert target.dropna().ge(0).all(), "Negative target"
assert target[valid].eq(panel.loc[valid, "observed_session_rows"]).all(), "Count mismatch"
assert target.eq(0).fillna(False).astype(bool).equals(panel["zero_is_assumed"]), "Inferred-zero flag mismatch"
assert len(panel) == audit["daily_rows"], "Row count differs from audit"
assert valid.sum() == audit["valid_targets"], "Valid-label count differs from audit"
assert panel.zero_is_assumed.sum() == audit["assumed_zero_targets"], "Zero count differs from audit"
print("Daily panel checks passed.")

In [ ]:
coverage = panel["coverage_status"].value_counts().rename("device_days").to_frame()
coverage["share_pct"] = 100 * coverage["device_days"] / len(panel)

label_status = pd.Series("Missing label", index=panel.index)
label_status.loc[valid] = "Positive count"
label_status.loc[panel["zero_is_assumed"]] = "Inferred zero"
labels = label_status.value_counts().rename("device_days").to_frame()
labels["share_pct"] = 100 * labels["device_days"] / len(panel)

display(coverage.round(2))
display(labels.round(2))

recommended = panel.loc[panel["recommended_period"]].copy()
splits = recommended.groupby("split").agg(
    calendar_rows=("com_id", "size"),
    usable_labels=("target_valid", "sum"),
    inferred_zeros=("zero_is_assumed", "sum"),
)
splits["missing_labels"] = splits["calendar_rows"] - splits["usable_labels"]
display(splits.loc[["train", "validation", "test"]])

## 2. Explore the data

Review label availability over the full calendar. Use the recommended training split to inspect the outcome distribution and weekday patterns. Monthly coverage here is the share of device-days with usable labels under the cleaning rules.

In [ ]:
by_month = panel.groupby(panel["reported_date"].dt.to_period("M"))["target_valid"].mean()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_month.index.astype(str), 100 * by_month, color="#267780")
ax.set(title="Share of usable labels by month", ylabel="Usable device-days (%)", ylim=(0, 100))
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
train = recommended.loc[recommended["split"].eq("train") & recommended["target_valid"]]
assert not train.empty, "No eligible training rows"
train_counts = train["sessions_target"]
# Group the upper tail for readability. This is a plotting choice.
plot_cap = max(1, int(train_counts.quantile(0.99)))
frequency = train_counts.clip(upper=plot_cap).value_counts().sort_index()

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_means = (
    train.groupby(train["reported_date"].dt.day_name())["sessions_target"]
    .mean().reindex(weekday_order).astype(float)
)

fig, (ax_dist, ax_week) = plt.subplots(1, 2, figsize=(12, 4))
ax_dist.bar(frequency.index.astype(int), frequency.to_numpy(), color="#267780")
ax_dist.set(title=f"Training counts ({plot_cap}+ grouped)", xlabel="Sessions per device-day",
            ylabel="Device-days")
ax_week.bar([day[:3] for day in weekday_order], weekday_means, color="#397967")
ax_week.set(title="Mean training count by weekday", ylabel="Sessions per device-day")
fig.tight_layout()
plt.show()

## 3. Preview historical inputs

Build the lags on the full continuous calendar before selecting usable rows. A missing day remains missing when it becomes a lag. The table below shows label and lag availability. Final model comparisons still need a shared prediction schedule, eligibility rules and matching evaluation rows.

In [ ]:
by_device = panel.groupby("com_id", sort=False)
for days in (1, 7):
    panel[f"lag_{days}"] = by_device["sessions_target"].shift(days)
    historical_date = by_device["reported_date"].shift(days)
    observed_date = historical_date.notna()
    assert (panel.loc[observed_date, "reported_date"] - historical_date[observed_date]).eq(
        pd.Timedelta(days=days)
    ).all()

panel["weekday"] = panel["reported_date"].dt.dayofweek
eligible = panel.loc[panel["recommended_period"] & panel["target_valid"]].copy()
eligible["both_lags_available"] = eligible[["lag_1", "lag_7"]].notna().all(axis=1)
readiness = eligible.groupby("split").agg(
    usable_labels=("sessions_target", "size"),
    rows_with_both_lags=("both_lags_available", "sum"),
)
display(readiness.loc[["train", "validation", "test"]])
display(eligible[["com_id", "reported_date", "sessions_target", "lag_1", "lag_7", "weekday"]].head())

## Team review

- Explain which dates and device-months account for masked labels. Discuss how inferred zeros could affect the evaluation.
- Interpret the weekday summary using the recommended training period.
- Agree on the forecast horizon, baseline formulas and missing-history rules before fitting models.
- Use `target_valid` and coverage fields for auditing and row selection. Treat `observed_session_rows` and `month_session_rows` as audit fields because they reveal same-day or later information.